# MetaGPT 角色定制與擴展

## 📚 簡介

本教程將教你如何創建自定義角色、定制行為，以及擴展 MetaGPT 的功能。

### 學習目標
- 理解 MetaGPT 的角色系統
- 創建自定義角色
- 定義自定義行為（Action）
- 配置角色間的協作
- 集成外部工具

## 1. 角色系統架構

### MetaGPT 角色的核心組件

```
Role (角色)
 ├── Profile (個人資料) - 角色描述
 ├── Goal (目標) - 角色目標
 ├── Constraints (約束) - 行為約束
 ├── Actions (行為列表) - 可執行的動作
 └── Memory (記憶) - 上下文記憶
```

In [ ]:
# 環境準備
import os
from dotenv import load_dotenv

load_dotenv()

# 導入核心模組
try:
    from metagpt.roles import Role
    from metagpt.actions import Action
    from metagpt.schema import Message
    from metagpt.logs import logger
    print("✓ MetaGPT 模組導入成功")
except ImportError as e:
    print(f"❌ 導入失敗: {e}")

## 2. 創建自定義 Action（行為）

Action 是角色執行的具體動作。讓我們先學習如何創建自定義 Action。

In [ ]:
# 示例 1: 創建簡單的 Action
from metagpt.actions import Action

class WriteCodeReview(Action):
    """
    自定義 Action: 代碼審查
    """
    
    name: str = "WriteCodeReview"
    
    async def run(self, code: str) -> str:
        """
        執行代碼審查
        
        Args:
            code: 要審查的代碼
            
        Returns:
            審查報告
        """
        prompt = f"""
        請審查以下代碼，提供改進建議:
        
        代碼:
        ```python
        {code}
        ```
        
        請從以下角度審查:
        1. 代碼質量
        2. 性能問題
        3. 安全漏洞
        4. 最佳實踐
        5. 可維護性
        """
        
        # 調用 LLM
        # result = await self.llm.aask(prompt)
        # return result
        
        # 示例返回（實際會調用 LLM）
        return "代碼審查報告：代碼質量良好，建議添加錯誤處理。"

print("✓ WriteCodeReview Action 已定義")

In [ ]:
# 示例 2: 帶參數的 Action
class GenerateDocumentation(Action):
    """
    自定義 Action: 生成文檔
    """
    
    name: str = "GenerateDocumentation"
    
    def __init__(self, doc_type: str = "API", **kwargs):
        super().__init__(**kwargs)
        self.doc_type = doc_type
    
    async def run(self, code: str) -> str:
        """
        生成文檔
        """
        prompt = f"""
        為以下代碼生成 {self.doc_type} 文檔:
        
        {code}
        
        請包含:
        - 功能描述
        - 參數說明
        - 返回值說明
        - 使用示例
        """
        
        return f"{self.doc_type} 文檔已生成"

print("✓ GenerateDocumentation Action 已定義")

## 3. 創建自定義 Role（角色）

現在讓我們創建自定義角色，並為其配置行為。

In [ ]:
# 示例 1: 創建代碼審查員角色
from metagpt.roles import Role
from metagpt.actions import Action

class CodeReviewer(Role):
    """
    代碼審查員角色
    負責審查代碼質量、提出改進建議
    """
    
    name: str = "CodeReviewer"
    profile: str = "代碼審查專家"
    goal: str = "確保代碼質量、發現潛在問題"
    constraints: str = "基於最佳實踐和編碼規範進行審查"
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        # 設置角色的行為
        self.set_actions([WriteCodeReview])
        
        # 設置監聽的消息類型（什麼類型的消息會觸發這個角色）
        # self._watch([WriteCode])  # 監聽 WriteCode 行為的輸出
    
    async def _act(self) -> Message:
        """
        角色的主要行為邏輯
        """
        logger.info(f"{self.name}: 開始審查代碼...")
        
        # 獲取要審查的代碼
        todo = self.rc.todo
        
        # 執行審查
        code = self.get_memories()  # 獲取上下文中的代碼
        review = await todo.run(code)
        
        # 返回審查結果
        msg = Message(
            content=review,
            role=self.profile,
            cause_by=type(todo)
        )
        
        return msg

print("✓ CodeReviewer Role 已定義")

In [ ]:
# 示例 2: 創建技術文檔撰寫員角色
class TechnicalWriter(Role):
    """
    技術文檔撰寫員
    負責撰寫技術文檔、API 文檔、使用指南
    """
    
    name: str = "TechnicalWriter"
    profile: str = "技術文檔專家"
    goal: str = "撰寫清晰、完整的技術文檔"
    constraints: str = "文檔應易於理解，包含示例和最佳實踐"
    
    def __init__(self, doc_types: list = None, **kwargs):
        super().__init__(**kwargs)
        
        # 配置文檔類型
        self.doc_types = doc_types or ["API", "User Guide", "README"]
        
        # 設置行為
        self.set_actions([GenerateDocumentation])
    
    async def _act(self) -> Message:
        """
        生成技術文檔
        """
        logger.info(f"{self.name}: 開始撰寫文檔...")
        
        # 獲取項目信息
        project_info = self.get_memories()
        
        # 生成各類文檔
        docs = []
        for doc_type in self.doc_types:
            action = GenerateDocumentation(doc_type=doc_type)
            doc = await action.run(project_info)
            docs.append(f"{doc_type}: {doc}")
        
        # 返回結果
        msg = Message(
            content="\n\n".join(docs),
            role=self.profile
        )
        
        return msg

print("✓ TechnicalWriter Role 已定義")

## 4. 複雜角色示例：DevOps 工程師

In [ ]:
# 定義 DevOps 相關的 Actions
class WriteDockerfile(Action):
    """生成 Dockerfile"""
    name: str = "WriteDockerfile"
    
    async def run(self, project_info: str) -> str:
        prompt = f"""
        為以下項目生成 Dockerfile:
        {project_info}
        
        要求:
        - 使用合適的基礎鏡像
        - 優化鏡像大小
        - 遵循最佳實踐
        """
        return "Dockerfile 已生成"

class WriteCICD(Action):
    """生成 CI/CD 配置"""
    name: str = "WriteCICD"
    
    async def run(self, project_info: str) -> str:
        prompt = f"""
        為項目生成 GitHub Actions workflow:
        {project_info}
        
        包含:
        - 代碼檢查
        - 測試運行
        - 構建和部署
        """
        return "CI/CD 配置已生成"

class WriteKubernetesConfig(Action):
    """生成 Kubernetes 配置"""
    name: str = "WriteKubernetesConfig"
    
    async def run(self, project_info: str) -> str:
        return "Kubernetes 配置已生成"

# 創建 DevOps 工程師角色
class DevOpsEngineer(Role):
    """
    DevOps 工程師
    負責容器化、CI/CD、部署配置
    """
    
    name: str = "DevOpsEngineer"
    profile: str = "DevOps 專家"
    goal: str = "實現自動化部署和運維"
    constraints: str = "遵循 DevOps 最佳實踐，確保可靠性和安全性"
    
    def __init__(self, enable_k8s: bool = False, **kwargs):
        super().__init__(**kwargs)
        
        # 根據配置設置行為
        actions = [WriteDockerfile, WriteCICD]
        if enable_k8s:
            actions.append(WriteKubernetesConfig)
        
        self.set_actions(actions)
    
    async def _act(self) -> Message:
        """
        執行 DevOps 任務
        """
        logger.info(f"{self.name}: 開始配置部署環境...")
        
        # 獲取項目信息
        project_info = self.get_memories()
        
        # 執行所有配置的 actions
        results = []
        for action_cls in self.actions:
            action = action_cls()
            result = await action.run(project_info)
            results.append(f"{action.name}: {result}")
        
        msg = Message(
            content="\n".join(results),
            role=self.profile
        )
        
        return msg

print("✓ DevOpsEngineer Role 已定義")

## 5. 角色協作配置

配置多個自定義角色協同工作。

In [ ]:
# 創建自定義團隊
async def create_custom_team():
    """
    創建包含自定義角色的團隊
    """
    from metagpt.team import Team
    from metagpt.roles import ProductManager, Engineer
    
    # 創建團隊
    team = Team()
    
    # 添加標準角色和自定義角色
    team.hire([
        ProductManager(),           # 產品經理
        Engineer(),                 # 工程師
        CodeReviewer(),            # 代碼審查員（自定義）
        TechnicalWriter(),         # 技術文檔員（自定義）
        DevOpsEngineer(enable_k8s=True)  # DevOps 工程師（自定義）
    ])
    
    print("團隊成員:")
    for role in team.roles:
        print(f"  • {role.name} - {role.profile}")
    
    # 定義項目需求
    idea = """
    創建一個 RESTful API 服務:
    - 用戶認證和授權
    - CRUD 操作
    - 使用 FastAPI
    - 需要 Docker 部署
    - 需要 CI/CD 配置
    """
    
    # 運行項目
    # team.invest(investment=8.0)
    # team.run_project(idea)
    # await team.run(n_round=5)
    
    print("\n提示: 團隊配置完成，取消註釋以運行")

# 運行示例
# await create_custom_team()

print("自定義團隊示例已加載")

## 6. 集成外部工具

將外部工具集成到 MetaGPT 角色中。

In [ ]:
# 示例: 集成代碼格式化工具
import subprocess
from typing import Optional

class FormatCode(Action):
    """
    使用 Black 格式化 Python 代碼
    """
    name: str = "FormatCode"
    
    async def run(self, code: str) -> str:
        """
        格式化代碼
        """
        try:
            # 使用 black 格式化代碼
            # 實際應用中會調用外部工具
            # result = subprocess.run(
            #     ['black', '-'],
            #     input=code.encode(),
            #     capture_output=True
            # )
            # formatted_code = result.stdout.decode()
            
            formatted_code = code  # 示例
            return formatted_code
        except Exception as e:
            logger.error(f"格式化失敗: {e}")
            return code

class RunLinter(Action):
    """
    使用 Pylint 進行代碼檢查
    """
    name: str = "RunLinter"
    
    async def run(self, code_path: str) -> str:
        """
        運行 linter
        """
        try:
            # result = subprocess.run(
            #     ['pylint', code_path],
            #     capture_output=True
            # )
            # return result.stdout.decode()
            
            return "Linter 檢查通過"
        except Exception as e:
            return f"Linter 檢查失敗: {e}"

# 創建代碼質量工程師角色
class CodeQualityEngineer(Role):
    """
    代碼質量工程師
    負責代碼格式化、靜態分析
    """
    
    name: str = "CodeQualityEngineer"
    profile: str = "代碼質量專家"
    goal: str = "確保代碼風格統一、符合規範"
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.set_actions([FormatCode, RunLinter])

print("✓ CodeQualityEngineer Role 已定義（集成外部工具）")

## 7. 高級特性：動態行為選擇

In [ ]:
# 創建智能角色，根據上下文選擇行為
class SmartEngineer(Role):
    """
    智能工程師
    根據項目類型和需求動態選擇行為
    """
    
    name: str = "SmartEngineer"
    profile: str = "全棧工程師"
    goal: str = "根據需求選擇最佳實現方案"
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        # 定義多種可選行為
        self.available_actions = {
            "web": ["WriteFlaskApp", "WriteReactApp"],
            "cli": ["WriteCLIApp"],
            "api": ["WriteAPIServer"],
            "data": ["WriteDataPipeline"]
        }
    
    async def _think(self) -> None:
        """
        分析需求，選擇合適的行為
        """
        # 分析需求類型
        requirement = self.get_memories()
        
        # 根據關鍵詞選擇行為
        project_type = "cli"  # 實際會通過 LLM 分析
        
        if "web" in requirement.lower() or "website" in requirement.lower():
            project_type = "web"
        elif "api" in requirement.lower() or "rest" in requirement.lower():
            project_type = "api"
        elif "data" in requirement.lower() or "pipeline" in requirement.lower():
            project_type = "data"
        
        # 設置對應的行為
        selected_actions = self.available_actions.get(project_type, ["WriteCLIApp"])
        logger.info(f"選擇項目類型: {project_type}, 行為: {selected_actions}")
        
        # 動態設置行為
        # self.set_actions(selected_actions)

print("✓ SmartEngineer Role 已定義（動態行為選擇）")

## 8. 完整示例：創建數據科學團隊

In [ ]:
# 數據科學相關的 Actions
class DataAnalysis(Action):
    """數據分析"""
    name: str = "DataAnalysis"
    async def run(self, data_desc: str) -> str:
        return "數據分析報告已生成"

class BuildMLModel(Action):
    """構建機器學習模型"""
    name: str = "BuildMLModel"
    async def run(self, requirements: str) -> str:
        return "ML 模型已構建"

class VisualizeData(Action):
    """數據可視化"""
    name: str = "VisualizeData"
    async def run(self, data: str) -> str:
        return "可視化圖表已生成"

# 數據科學家角色
class DataScientist(Role):
    """數據科學家"""
    name: str = "DataScientist"
    profile: str = "數據科學專家"
    goal: str = "分析數據、構建模型、提供洞察"
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.set_actions([DataAnalysis, BuildMLModel, VisualizeData])

# 數據工程師角色
class DataEngineer(Role):
    """數據工程師"""
    name: str = "DataEngineer"
    profile: str = "數據工程專家"
    goal: str = "構建數據管道、處理數據"

print("✓ 數據科學團隊角色已定義")
print("\n可用角色:")
print("  • DataScientist - 數據科學家")
print("  • DataEngineer - 數據工程師")

## 📝 總結

本教程中，我們學習了:

### 核心技能
1. ✅ 創建自定義 Action（行為）
2. ✅ 創建自定義 Role（角色）
3. ✅ 配置角色協作
4. ✅ 集成外部工具
5. ✅ 實現動態行為選擇

### 實踐案例
- CodeReviewer - 代碼審查員
- TechnicalWriter - 技術文檔員
- DevOpsEngineer - DevOps 工程師
- CodeQualityEngineer - 代碼質量工程師
- DataScientist - 數據科學家

### 關鍵要點
- Action 定義具體的執行邏輯
- Role 組合多個 Actions
- 可以根據需求靈活組合角色
- 支持集成外部工具和服務
- 可以實現複雜的決策邏輯

## 🎯 下一步

- **3.實戰項目.ipynb**: 使用自定義角色完成實際項目
- **4.高級特性.ipynb**: 更多進階技巧

## 💡 最佳實踐

1. **單一職責**: 每個 Action 只做一件事
2. **清晰命名**: 使用描述性的名稱
3. **錯誤處理**: 妥善處理異常情況
4. **文檔完整**: 為自定義角色添加詳細註釋
5. **測試驗證**: 在小範圍測試新角色

祝你創造出強大的自定義角色！🚀